# Herning — envelope model check

Checks the multiplicative model for AE and UL discussed on 2026-10-29. Per frequency band $b$,

$$x_b(t) = \sigma_b(t)\,g_b(t),$$

with $g_b$ a stationary narrowband noise (the carrier) and $\sigma_b \ge 0$ a slowly varying intensity.
The band energy averaged over blocks of length $T$ is then $E_b[n] \propto \sigma_b^2(nT)\,\chi_b[n]$, and

$$L_b[n] = \ln E_b[n] = \ln\sigma_b^2(nT) + \ln\chi_b[n].$$

If the carrier is Gaussian, $\chi_b$ is approximately gamma distributed with a shape $a_b$ that follows from the
band's own spectrum and $T$ (about $B_b T$ for wide bands, about 1 for bands narrower than $1/T$). So the
fluctuation of $L_b$ that the carrier alone produces is known — the **null** — and anything beyond it is a
change of intensity (rotation-locked modulation or bursts).

Three checks:
1. **Variance.** Noise reference (bearing not rotating): does $L_b$ fluctuate as the null predicts? Rotating: how much
   variance is there beyond the null, per band?
2. **Distribution.** How is the excess distributed? A heavy upper tail means bursts; a symmetric widening means smooth modulation.
3. **Shared intensity.** Do the bands (AE and UL together) move together, i.e. is there one common $\sigma(t)$?

Not part of any pipeline and decides nothing (D68). Settings approved for this notebook (2026-10-29): bands as the feature
baseline (D60/D61), blocks of 125 samples = 20 kHz (as `herning_eda.ipynb` section 8), sweeps = section 5 picks plus the
noise reference at 40/60/100 °C (five each), residual = $L_b$ minus the band level only. Other choices made only for
looking are marked *EDA choice*.

**Run:** `uv sync --group eda`, then open this notebook. It reads 40 sweeps (about a minute).

Sections: 1. Setup  2. Sweeps  3. Bands  4. Band energies  5. Check 1: variance  6. Check 2: distribution
7. Check 3: shared intensity  8. Reading the results

## 1. Setup

In [ ]:
import sys
from pathlib import Path

import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from scipy.special import polygamma
from scipy.stats import loggamma

from harness import kappa
from harness.data import herning


def repo_root():
    """The repository root: the first folder upwards that contains pyproject.toml."""
    p = Path.cwd().resolve()
    while not (p / "pyproject.toml").exists():
        p = p.parent
    return p


ROOT = repo_root()
sys.path.insert(0, str(ROOT / "eda" / "herning"))
from check_bands import role  # sweep roles exactly as in the band check (D52, D55)

BAND_CFG = yaml.safe_load(open(ROOT / "configs" / "band_check_herning.yaml"))
OP = BAND_CFG["operating_point"]  # D55
f = h5py.File(ROOT / BAND_CFG["source"], "r")
SWEEPS = f["sweeps"]
FS = 2.5e6  # scope sample rate, both channels

# Plot colours as in herning_eda.ipynb.
COLOR = {"AE": "#2a78d6", "UL": "#eb6834"}
MUTED = "#52514e"
plt.rcParams.update({"figure.dpi": 110, "font.size": 9, "axes.spines.top": False, "axes.spines.right": False})

## 2. Sweeps
Sweep table from the telemetry, as in `herning_eda.ipynb` section 3 (rpm and temperature as D55, roles and κ as the
band check). Groups: the section 5 picks plus the noise reference (rpm target 100: VFD on, bearing not rotating).

In [ ]:
rows = []
for name in herning.sweep_names(f):  # time order
    a = SWEEPS[name].attrs
    rpm = float(a["telem_vfd_cmd_hz"]) * OP["rpm_per_cmd_hz"]
    temp = float(a[OP["temperature_field"]])
    r = role(float(a["telem_rpm_target"]), float(a["telem_temp_target"]), rpm, temp, OP)
    rows.append({"sweep": name, "rpm_target": float(a["telem_rpm_target"]), "temp_target": float(a["telem_temp_target"]),
                 "rpm": rpm, "temperature_c": temp, "role": r,
                 "kappa": kappa.kappa(rpm, temp, OP["d_pw_mm"], OP["nu_40_cst"], OP["nu_100_cst"]) if r == "analysed" else np.nan})
sweeps = pd.DataFrame(rows).set_index("sweep")


def describe(name):
    r = sweeps.loc[name]
    return f"{name}: {r.rpm:.0f} rpm, {r.temperature_c:.0f} °C, κ={r.kappa:.3f}, {r.role}"


# Groups as (rpm target, temperature target), approved 2026-10-29.
GROUPS = {"ref 40 °C": (100, 40), "ref 60 °C": (100, 60), "ref 100 °C": (100, 100),
          "500 rpm 60 °C": (500, 60), "1500 rpm 60 °C": (1500, 60), "3000 rpm 60 °C": (3000, 60),
          "1500 rpm 40 °C": (1500, 40), "1500 rpm 100 °C": (1500, 100)}
N_PER_GROUP = 5
FIRST = 2  # EDA choice: start at the third sweep of each (rpm, temperature) target, as section 5 (k=2)

picks = {}
for g, (r, t) in GROUPS.items():
    names = sweeps.index[(sweeps.rpm_target == r) & (sweeps.temp_target == t)]
    picks[g] = list(names[FIRST:FIRST + N_PER_GROUP])
    print(f"{g:16s} {len(picks[g])} of {len(names)} sweeps, roles: {', '.join(sweeps.loc[picks[g], 'role'].unique())}")

## 3. Bands
Bands as the feature baseline: AE octaves 50–100–200–400–560 kHz (D61); UL base-10 octaves with centres 1.0 … 126 kHz,
edges at centre·10^(±0.15) (D60).

Each band is cut out of the sweep's FFT with a mask whose edges are short cosine transitions. At every inner edge the
falling cos of the lower band and the rising sin of the upper band have squares that add to 1 (power complementary), so
the band powers of a channel add up to the channel's power between the outer edges. The table gives each band's width $B$,
the carrier's correlation time $1/B$, and $B\cdot T$ for $T$ = 50 µs.

In [ ]:
BLOCK = 125  # samples per block: T = 50 us, envelope rate 20 kHz (approved 2026-10-29, as herning_eda section 8)
T = BLOCK / FS
TRANSITION = 0.1  # EDA choice: width of the cosine transition at each band edge, as a fraction of the edge frequency

EDGES = {"AE": [50e3, 100e3, 200e3, 400e3, 560e3],                     # D61
         "UL": [1e3 * 10 ** (0.3 * k - 0.15) for k in range(9)]}       # D60: 0.71, 1.41, ..., 178 kHz
BANDS = [(ch, lo, hi) for ch in EDGES for lo, hi in zip(EDGES[ch][:-1], EDGES[ch][1:])]
LABELS = [f"{ch} {lo / 1e3:.3g}–{hi / 1e3:.3g}" for ch, lo, hi in BANDS]  # kHz


def masks(freqs, edges, w=TRANSITION):
    """Amplitude responses (on the grid `freqs`) of the contiguous bands between `edges`.
    Around an edge e the response rises (sin) or falls (cos) over [e(1 - w/2), e(1 + w/2)]."""
    def s(e):  # 0 below the transition, 1 above, linear in between
        return np.clip((freqs - e * (1 - w / 2)) / (e * w), 0, 1)
    return [np.sin(np.pi / 2 * s(lo)) * np.cos(np.pi / 2 * s(hi)) for lo, hi in zip(edges[:-1], edges[1:])]


N = len(SWEEPS[picks["ref 60 °C"][0]]["AE"]["voltage"])  # samples per sweep (500,000)
FREQS = np.fft.rfftfreq(N, 1 / FS)
MASKS = {ch: masks(FREQS, EDGES[ch]) for ch in EDGES}

pd.DataFrame({"band (kHz)": LABELS, "B (kHz)": [(hi - lo) / 1e3 for _, lo, hi in BANDS],
              "1/B (µs)": [1e6 / (hi - lo) for _, lo, hi in BANDS],
              "B·T": [(hi - lo) * T for _, lo, hi in BANDS]}).set_index("band (kHz)").round(3)

In [ ]:
# The squared masks per channel and their sum (1 between the outer edges, apart from the outer transitions).
fig, ax = plt.subplots(1, 2, figsize=(12, 2.8))
for a, ch in zip(ax, EDGES):
    for m in MASKS[ch]:
        a.semilogx(FREQS[1:], m[1:] ** 2, lw=0.8, color=COLOR[ch])
    a.semilogx(FREQS[1:], sum(m ** 2 for m in MASKS[ch])[1:], lw=1, ls="--", color=MUTED, label="sum of squares")
    a.set_xlim(300, 1e6); a.set_xlabel("frequency (Hz)"); a.set_ylabel(f"{ch}: squared response"); a.legend(frameon=False)
plt.tight_layout()

## 4. Band energies
For every band: the analytic band signal $z_b$ (the sweep's positive-frequency spectrum times the mask, doubled, inverse FFT),
its instantaneous power $p_b = |z_b|^2/2$ (whose mean is the band's power), and the block means $E_b[n]$ of $p_b$ over 125 samples.

**Null.** For a stationary Gaussian carrier with the band's spectrum, $E_b$ is approximately gamma distributed (moments matched)
with shape $a_b = 1/r_b$, where $r_b = \mathrm{Var}(E_b)/\mathrm{E}(E_b)^2 = \frac{1}{M}\sum_{|j|<M}(1-|j|/M)\,|\rho_b(j)|^2$,
$M$ = 125 and $\rho_b$ the autocorrelation of $z_b$. $\rho_b$ is taken from the sweep's own band spectrum, so the null uses the
carrier's actual spectral shape (lines included) and nothing else. Then $\mathrm{Var}(\ln E_b) = \psi'(a_b)$ (trigamma).

In [ ]:
def band_energies(name):
    """Block energies E (bands x blocks, V^2) of one sweep and the null's gamma shape a (bands)."""
    E, a = [], []
    lags = np.arange(-BLOCK + 1, BLOCK)  # negative lags index from the end of the circular autocorrelation
    for ch in EDGES:
        v = SWEEPS[name][ch]["voltage"][:]
        X = np.fft.rfft(v - v.mean())
        for m in MASKS[ch]:
            Z = np.zeros(N, complex)
            Z[: len(X)] = 2 * X * m          # positive frequencies only (the masks are 0 at DC and Nyquist)
            z = np.fft.ifft(Z)               # analytic band signal
            p = np.abs(z) ** 2 / 2           # instantaneous power
            E.append(p[: N // BLOCK * BLOCK].reshape(-1, BLOCK).mean(axis=1))
            R = np.fft.ifft(np.abs(Z) ** 2)  # circular autocorrelation of z (unnormalised)
            rho = R[lags] / R[0]
            a.append(BLOCK / np.sum((1 - np.abs(lags) / BLOCK) * np.abs(rho) ** 2))  # a = 1 / r
    return np.array(E), np.array(a)


results = {}  # sweep name -> group, L = ln E (bands x blocks), a (bands)
for g, names in picks.items():
    for name in names:
        E, a = band_energies(name)
        results[name] = {"group": g, "L": np.log(E), "a": a}
print(len(results), "sweeps;", results[name]["L"].shape[1], "blocks each")

In [ ]:
# One noise-reference sweep and one 3000 rpm sweep: L_b minus its sweep mean, all bands (rows) over the sweep.
# Dotted lines (3000 rpm): one per revolution, arbitrary phase (no shaft reference in the data).
SHOW = [picks["ref 60 °C"][0], picks["3000 rpm 60 °C"][0]]  # EDA choice
t0 = float(SWEEPS[SHOW[0]]["AE"].attrs["x_origin"])  # sweeps start at about -100 ms
fig, ax = plt.subplots(2, 1, figsize=(12, 6.5), sharex=True)
for a_, name in zip(ax, SHOW):
    L = results[name]["L"]
    t_ms = 1e3 * (t0 + (np.arange(L.shape[1]) + 0.5) * T)
    im = a_.pcolormesh(t_ms, np.arange(len(BANDS)), L - L.mean(axis=1, keepdims=True), cmap="RdBu_r",
                       vmin=-3, vmax=3, shading="nearest")  # EDA choice: colour range +-3 (ln units)
    a_.set_yticks(range(len(BANDS)), LABELS, fontsize=7)
    a_.set_title(describe(name), loc="left", fontsize=9)
    plt.colorbar(im, ax=a_, label="ln E − sweep mean")
rpm = sweeps.loc[SHOW[1], "rpm"]
for x in np.arange(t_ms[0], t_ms[-1], 60e3 / rpm):
    ax[1].axvline(x, color="k", lw=0.5, ls=":")
ax[-1].set_xlabel("time (ms)")
plt.tight_layout()

In [ ]:
# The same two sweeps, two bands as lines (EDA choice: which bands).
LINE_BANDS = ["AE 100–200", "UL 11.2–22.4"]
fig, ax = plt.subplots(len(LINE_BANDS), 1, figsize=(12, 4.5), sharex=True)
for a_, lab in zip(ax, LINE_BANDS):
    b = LABELS.index(lab)
    for name, c in zip(SHOW, [MUTED, COLOR[lab[:2]]]):
        L = results[name]["L"][b]
        a_.plot(t_ms, L - L.mean(), lw=0.5, color=c, label=describe(name).split(": ")[1].split(", κ")[0])
    a_.set_ylabel(f"{lab} kHz\nln E − mean")
ax[0].legend(frameon=False, fontsize=7); ax[-1].set_xlabel("time (ms)")
plt.tight_layout()

## 5. Check 1: variance beyond the null
Per sweep and band: the observed variance of $L_b$ over the sweep divided by the null variance $\psi'(a_b)$. Table and figure:
median over the five sweeps of a group.
- ratio ≈ 1: the band fluctuates exactly as a stationary Gaussian carrier would;
- ratio > 1: intensity changes (modulation, bursts). $1 - 1/\text{ratio}$ is the share of $\mathrm{Var}(L_b)$ that is intensity;
- ratio < 1: less fluctuation than Gaussian noise, e.g. deterministic lines (a sinusoid has a constant envelope; VFD comb, QUESTIONS.md Q3).

In [ ]:
rows = []
for name, r in results.items():
    for b, lab in enumerate(LABELS):
        null_var = polygamma(1, r["a"][b])  # Var(ln E) under the null: trigamma(a)
        rows.append({"group": r["group"], "sweep": name, "band": lab, "a": r["a"][b],
                     "null_var": null_var, "obs_var": r["L"][b].var(), "ratio": r["L"][b].var() / null_var})
check1 = pd.DataFrame(rows)

shape = check1.groupby("band").a.median().reindex(LABELS).rename("null shape a (median)")
ratio = check1.groupby(["band", "group"]).ratio.median().unstack("group").reindex(index=LABELS, columns=list(GROUPS))
pd.concat([shape, ratio], axis=1).round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 3.8))
greys, blues = plt.get_cmap("Greys"), plt.get_cmap("Blues")
for i, g in enumerate(GROUPS):
    is_ref = g.startswith("ref")
    c = greys(0.4 + 0.25 * i) if is_ref else blues(0.35 + 0.65 * (i - 3) / (len(GROUPS) - 4))
    ax.semilogy(range(len(LABELS)), ratio[g], "o-" if is_ref else "s-", ms=4, lw=1, color=c, label=g)
ax.axhline(1, color=MUTED, lw=0.8, ls="--")
ax.set_xticks(range(len(LABELS)), LABELS, rotation=45, ha="right", fontsize=7)
ax.set_ylabel("Var(ln E) / null variance"); ax.legend(frameon=False, fontsize=7, ncol=2)
plt.tight_layout()

## 6. Check 2: distribution of the residual
Residual = $L_b$ minus the band level only (approved 2026-10-29), pooled over the five sweeps of a group. Each sweep is
aligned by its **median** (robust to bursts) to the null's median. Grey curve: the null log-gamma density for the
group's median shape $a_b$. Log density axis, so the tails are visible.

In [ ]:
HIST_BANDS = ["AE 100–200", "AE 200–400", "UL 11.2–22.4", "UL 44.7–89.1"]  # EDA choice
HIST_GROUPS = ["ref 60 °C", "500 rpm 60 °C", "3000 rpm 60 °C"]         # EDA choice
HIST_COLORS = [MUTED, "#9fc3ee", "#2a78d6"]


def residual(name, b):
    """L_b of one sweep, shifted so that its median equals the null's median (null: ln of a gamma(a) variable)."""
    L, a = results[name]["L"][b], results[name]["a"][b]
    return L - np.median(L) + loggamma(a).median()


fig, ax = plt.subplots(1, len(HIST_BANDS), figsize=(13, 3.4), sharey=True)
for a_, lab in zip(ax, HIST_BANDS):
    b = LABELS.index(lab)
    for g, c in zip(HIST_GROUPS, HIST_COLORS):
        y = np.concatenate([residual(n, b) for n in picks[g]])
        a_.hist(y, bins=80, density=True, histtype="step", lw=1, color=c, label=g)
    a_med = np.median([results[n]["a"][b] for g in HIST_GROUPS for n in picks[g]])
    grid = np.linspace(*a_.get_xlim(), 400)
    a_.plot(grid, loggamma(a_med).pdf(grid), color="k", lw=0.8, ls="--", label=f"null, a={a_med:.2f}")
    a_.set_yscale("log"); a_.set_ylim(1e-4, None); a_.set_title(f"{lab} kHz", loc="left", fontsize=9)
    a_.set_xlabel("ln E (aligned)")
ax[0].set_ylabel("density"); ax[-1].legend(frameon=False, fontsize=7)
plt.tight_layout()

In [ ]:
# Upper tail: share of blocks above the null's 99.9 % quantile (0.001 under the null), mean over a group's sweeps.
TAIL_Q = 0.999  # EDA choice
tail = {}
for g, names in picks.items():
    tail[g] = [np.mean([np.mean(residual(n, b) > loggamma(results[n]["a"][b]).ppf(TAIL_Q)) for n in names])
               for b in range(len(LABELS))]
pd.DataFrame(tail, index=LABELS).round(4)

## 7. Check 3: shared intensity across bands
Per sweep: the correlation matrix of $L_b$ across the 12 bands (AE and UL together, same clock). Under the null the bands are
nearly independent (disjoint frequency bands of a stationary Gaussian process), so the matrix is close to the identity and all
eigenvalues are close to 1. A common intensity $\sigma(t)$ gives positive correlations everywhere and one large eigenvalue.
Correlations of noisy bands (small $a_b$) are attenuated: by $\sqrt{s_b s_c}$, where $s_b = 1 - 1/\text{ratio}$ is the intensity
share from Check 1.

In [ ]:
CORR_GROUPS = ["ref 60 °C", "500 rpm 60 °C", "3000 rpm 60 °C"]  # EDA choice
fig, ax = plt.subplots(1, len(CORR_GROUPS), figsize=(13, 4))
for a_, g in zip(ax, CORR_GROUPS):
    C = np.mean([np.corrcoef(results[n]["L"]) for n in picks[g]], axis=0)  # mean over the group's sweeps
    im = a_.imshow(C, cmap="RdBu_r", vmin=-1, vmax=1)
    a_.set_xticks(range(len(LABELS)), LABELS, rotation=90, fontsize=6)
    a_.set_yticks(range(len(LABELS)), LABELS, fontsize=6)
    a_.set_title(g, loc="left", fontsize=9)
plt.colorbar(im, ax=ax, shrink=0.8, label="correlation of ln E")

In [ ]:
# Eigenvalues of each sweep's correlation matrix (largest first), median per group; and the leading
# eigenvector's loadings per band (sign chosen so the loadings sum to a positive number), median per group.
fig, ax = plt.subplots(1, 2, figsize=(12, 3.6))
for i, g in enumerate(GROUPS):
    is_ref = g.startswith("ref")
    c = greys(0.4 + 0.25 * i) if is_ref else blues(0.35 + 0.65 * (i - 3) / (len(GROUPS) - 4))
    eig, load = [], []
    for n in picks[g]:
        w, V = np.linalg.eigh(np.corrcoef(results[n]["L"]))
        eig.append(w[::-1])
        v1 = V[:, -1]
        load.append(v1 * np.sign(v1.sum()))
    ax[0].plot(range(1, len(LABELS) + 1), np.median(eig, axis=0), "o-" if is_ref else "s-", ms=3, lw=1, color=c, label=g)
    ax[1].plot(range(len(LABELS)), np.median(load, axis=0), "o-" if is_ref else "s-", ms=3, lw=1, color=c)
ax[0].axhline(1, color=MUTED, lw=0.8, ls="--"); ax[0].set_xlabel("eigenvalue rank"); ax[0].set_ylabel("eigenvalue")
ax[0].legend(frameon=False, fontsize=7, ncol=2)
ax[1].axhline(0, color=MUTED, lw=0.8); ax[1].set_ylabel("leading eigenvector loading")
ax[1].set_xticks(range(len(LABELS)), LABELS, rotation=45, ha="right", fontsize=7)
plt.tight_layout()

In [ ]:
# The two largest eigenvalues per group (median over sweeps). Under the null both are close to 1;
# lambda_1 = 12 would mean all bands move together perfectly.
pd.DataFrame({g: np.median([np.linalg.eigvalsh(np.corrcoef(results[n]["L"]))[::-1][:2] for n in picks[g]], axis=0)
              for g in GROUPS}, index=["lambda_1", "lambda_2"]).T.round(2)

## 8. Reading the results
- **Check 1 on the noise reference** tests the model itself: ratios near 1 mean the stationary-Gaussian carrier describes the bands
  when nothing rotates, so the null is calibrated. Ratios below 1 point to deterministic lines; above 1 to fluctuations that
  are not the bearing (the bearing does not rotate), e.g. electrical.
- **Check 1 on rotating sweeps**: bands with ratios well above 1 carry intensity changes within a sweep, i.e. information a
  time-resolved envelope has beyond the band level (what the MSSP statistics see). Bands near 1 carry, at this block length,
  only their level.
- **Check 2**: an upper tail above the null while the bulk follows it means sparse bursts; a bulk that is wider on both sides means
  smooth modulation. The tail table gives a burst rate per band (relative to 0.001).
- **Check 3**: one dominant eigenvalue with same-sign loadings over AE and UL bands supports a single common intensity
  $\sigma(t)$ (strong compression possible); several sizeable eigenvalues mean different bands carry different temporal information.
  Structure that also appears in the noise reference is not from the bearing.
- Not done here: separating the rotation-locked part from bursts (residual = band level only, as approved), other block lengths.